In [24]:
import numpy as np 
import pandas as pd 
import json


In [25]:
with open ("filePath.json", "r") as file:
    content = json.load(file)
    file_path_pep = content["PEP_3"]
    file_path_ko = content["KO_3"]
    file_path_pkl = content["pairs_data"]


In [26]:
df_pep = pd.read_csv(file_path_pep)
df_ko = pd.read_csv(file_path_ko)


In [27]:
df = pd.DataFrame()


In [28]:
df_pep["Date"] = pd.to_datetime(df_pep["Date"])
df_ko["Date"] = pd.to_datetime(df_ko["Date"])
df_pep["Normalized_Close"] = df_pep["Close"] / df_pep["Close"].iloc[0]
df_ko["Normalized_Close"] = df_ko["Close"] / df_ko["Close"].iloc[0]
df_pep["Daily_Return"] = df_pep["Close"].pct_change()
df_ko["Daily_Return"] = df_ko["Close"].pct_change()


In [29]:
# A = KO | B = PEP
# PEP - KO --> +  = pep is performing better, - = KO performing better
df["Date"] = df_pep["Date"]
df["Spread"] = df_pep["Normalized_Close"] - df_ko["Normalized_Close"]
df["Spread_Mean"] = df["Spread"].rolling(window=20).mean()
df["Spread_Std"] = df["Spread"].rolling(window=20).std()
df["Z_Score"] = (df["Spread"]-df["Spread_Mean"]) / df["Spread_Std"]
df["Daily_A_Norm"] = df_ko["Normalized_Close"]
df["Daily_A"] = df_ko["Daily_Return"]
df["Daily_B_Norm"] = df_pep["Normalized_Close"]
df["Daily_B"] = df_pep["Daily_Return"]
stock_return = 0.5*df_pep["Daily_Return"] + 0.5*df_ko["Daily_Return"]
df["Cum_Stock"] = (1+stock_return.fillna(0)).cumprod()


Z_Score = 0: No trade. Stocks moving normal relative to 20-Day avg.
Z_Score > 2: Long KO b/c pep is outperforming it, so believe KO will snap back up.
Z_Score < -2: Long PEP b/c KO is outperforming it, so believe PEP will snap back up.

In [31]:
df.to_pickle(file_path_pkl)
